In [ ]:
from itertools import combinations
import numpy as np
from scipy.sparse import coo_matrix

def popcount(x):
    return x.bit_count()

def occ(b, p):
    return (b >> p) & 1

def parity_before(b, p):
    mask = (1 << p) - 1
    return (b & mask).bit_count() % 2

def annihilate(b, p):
    if not occ(b, p):
        return None
    sign = -1 if parity_before(b, p) else 1
    return sign, b ^ (1 << p)

def create(b, p):
    if occ(b, p):
        return None
    sign = -1 if parity_before(b, p) else 1
    return sign, b | (1 << p)

def apply_one_body(b, p, q):
    # a_p^\dagger a_q |b>
    result = annihilate(b, q)
    if result is None:
        return None
    s1, b1 = result

    result = create(b1, p)
    if result is None:
        return None
    s2, b2 = result

    return s1 * s2, b2

def apply_two_body(b, p, q, r, s):
    # a_p^\dagger a_q^\dagger a_s a_r |b>
    result = annihilate(b, r)
    if result is None:
        return None
    s1, b1 = result

    result = annihilate(b1, s)
    if result is None:
        return None
    s2, b2 = result

    result = create(b2, q)
    if result is None:
        return None
    s3, b3 = result

    result = create(b3, p)
    if result is None:
        return None
    s4, b4 = result

    return s1 * s2 * s3 * s4, b4

In [4]:
Det = int

hf = Det(0b1101)
hf.bit_count()

3

In [5]:
def fixed_N_basis(M, N):
    basis = []
    for occs in combinations(range(M), N):
        b = 0
        for p in occs:
            b |= 1 << p
        basis.append(b)
    return basis

In [ ]:
def build_fci_jw_sparse(M, N, h, g, E_nuc=0.0, tol=1e-14):
    """
    h[p,q] = one-electron spin-orbital integral
    g[p,q,r,s] = two-electron integral <pq|rs>
    
    Hamiltonian:
    H = sum_pq h[p,q] a_p^dag a_q
      + 1/2 sum_pqrs g[p,q,r,s] a_p^dag a_q^dag a_s a_r
      + E_nuc
    """

    basis = fixed_N_basis(M, N)
    index = {b: i for i, b in enumerate(basis)}

    rows = []
    cols = []
    vals = []

    for col, b in enumerate(basis):

        # nuclear repulsion
        if abs(E_nuc) > tol:
            rows.append(col)
            cols.append(col)
            vals.append(E_nuc)

        # one-electron terms
        for p in range(M):
            for q in range(M):
                if abs(h[p, q]) < tol:
                    continue

                result = apply_one_body(b, p, q)
                if result is None:
                    continue

                phase, b_new = result
                row = index.get(b_new)

                if row is not None:
                    rows.append(row)
                    cols.append(col)
                    vals.append(phase * h[p, q])

        # two-electron terms
        for p in range(M):
            for q in range(M):
                for r in range(M):
                    for s in range(M):
                        val = 0.5 * g[p, q, r, s]
                        if abs(val) < tol:
                            continue

                        result = apply_two_body(b, p, q, r, s)
                        if result is None:
                            continue

                        phase, b_new = result
                        row = index.get(b_new)

                        if row is not None:
                            rows.append(row)
                            cols.append(col)
                            vals.append(phase * val)

    H = coo_matrix((vals, (rows, cols)), shape=(len(basis), len(basis)))
    return H.tocsr(), basis